# 06 — Model Dataset Split
**Fingo Income Predictor** | Tim CC26-PSU217

Input: `data/processed/income_features.csv`  
Output:
- `outputs/model_contract/income_train.csv`
- `outputs/model_contract/income_val.csv`
- `outputs/model_contract/income_test.csv`
- `outputs/model_contract/income_scalers.pkl`
- `outputs/model_contract/feature_columns.json`
- `outputs/model_contract/model_contract.json`

**Split:** Kronologis by `synthetic_user_id` — bukan random row.

Notebook ini membuat dataset final untuk AI Engineer, melakukan split train/validation/test berbasis user agar tidak ada overlap user antar subset, lalu menyimpan scaler dan model contract.

In [10]:
# GIT PULL — Sinkronisasi terbaru dari remote sebelum mulai
import os, shutil, subprocess

try:
    from google.colab import userdata
except Exception:
    userdata = None

os.chdir("/content")

GITHUB_USERNAME = "ClarisyaA"
REPO_NAME       = "fingo-income-analysis"
BRANCH_NAME     = "feat/income-predictor-final"
LOCAL_DIR       = f"/content/{REPO_NAME}"
FRESH_CLONE     = False  # Set True hanya kalau mau clone ulang dari nol

def get_remote_url():
    try:
        token = userdata.get("GITHUB_TOKEN") if userdata else os.environ.get("GITHUB_TOKEN", "")
        if token:
            return f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git", token
    except Exception:
        pass
    return f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git", None

remote_url, token = get_remote_url()

def mask_cmd(cmd):
    return cmd.replace(token, "***TOKEN***") if token else cmd

def run_cmd(cmd, check=True, cwd="/content"):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    print(f"$ {mask_cmd(cmd)}")
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command gagal: {mask_cmd(cmd)}")
    return r

def remote_branch_exists():
    r = run_cmd(f"git ls-remote --heads {remote_url} {BRANCH_NAME}", check=False)
    return r.stdout.strip() != ""

branch_exists = remote_branch_exists()

if FRESH_CLONE and os.path.exists(LOCAL_DIR):
    os.chdir("/content")
    shutil.rmtree(LOCAL_DIR)

if not os.path.exists(LOCAL_DIR):
    if branch_exists:
        run_cmd(f"git clone -b {BRANCH_NAME} {remote_url} {LOCAL_DIR}")
    else:
        run_cmd(f"git clone {remote_url} {LOCAL_DIR}")
        run_cmd(f"git checkout -b {BRANCH_NAME}", cwd=LOCAL_DIR)
else:
    run_cmd(f"git remote set-url origin {remote_url}", cwd=LOCAL_DIR)
    run_cmd("git fetch origin", cwd=LOCAL_DIR)
    if branch_exists:
        local_b = run_cmd(f"git branch --list {BRANCH_NAME}", check=False, cwd=LOCAL_DIR).stdout.strip()
        if local_b:
            run_cmd(f"git checkout {BRANCH_NAME}", cwd=LOCAL_DIR)
        else:
            run_cmd(f"git checkout -b {BRANCH_NAME} origin/{BRANCH_NAME}", cwd=LOCAL_DIR)
        run_cmd(f"git pull --rebase origin {BRANCH_NAME}", cwd=LOCAL_DIR)
    else:
        current_branch = run_cmd("git branch --show-current", check=False, cwd=LOCAL_DIR).stdout.strip()
        if current_branch != BRANCH_NAME:
            local_b = run_cmd(f"git branch --list {BRANCH_NAME}", check=False, cwd=LOCAL_DIR).stdout.strip()
            if local_b:
                run_cmd(f"git checkout {BRANCH_NAME}", cwd=LOCAL_DIR)
            else:
                run_cmd(f"git checkout -b {BRANCH_NAME}", cwd=LOCAL_DIR)

os.chdir(LOCAL_DIR)
run_cmd(f"git remote set-url origin {remote_url}", cwd=LOCAL_DIR)
print("\nRepo siap digunakan")
print(f"Working directory: {os.getcwd()}")
run_cmd("git branch --show-current", cwd=LOCAL_DIR)
run_cmd("git status --short", check=False, cwd=LOCAL_DIR)


$ git ls-remote --heads https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git feat/income-predictor-final
35d5b6603cf5f9d8db00e0d94dc11d15529885c1	refs/heads/feat/income-predictor-final
$ git remote set-url origin https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git
$ git fetch origin
$ git branch --list feat/income-predictor-final
* feat/income-predictor-final
$ git checkout feat/income-predictor-final
Your branch is up to date with 'origin/feat/income-predictor-final'.
Already on 'feat/income-predictor-final'
$ git pull --rebase origin feat/income-predictor-final
Already up to date.
From https://github.com/ClarisyaA/fingo-income-analysis
 * branch            feat/income-predictor-final -> FETCH_HEAD
$ git remote set-url origin https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git

Repo siap digunakan
Working directory: /content/fingo-income-analysis
$ git branch --show-current
feat/income-predictor-final
$ git status --short


CompletedProcess(args='git status --short', returncode=0, stdout='', stderr='')

In [11]:
# CELL 06.2 — Setup
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'scikit-learn', '--quiet'])

import os, json, pickle, warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
np.random.seed(42)
os.chdir("/content/fingo-income-analysis")

DIRECTION_THRESHOLD = 0.10

ORDERED_GIG_TYPES = [
    'ojek_online', 'kurir', 'jualan_online', 'freelance_desain',
    'freelance_it', 'content_creator', 'tutor', 'pekerja_harian'
]

def ensure_dir(path):
    d = os.path.dirname(path) if '.' in os.path.basename(path) else path
    if d:
        os.makedirs(d, exist_ok=True)

def safe_to_csv(df, path, **kwargs):
    ensure_dir(path)
    df.to_csv(path, index=kwargs.pop('index', False), **kwargs)

print('Setup selesai')


Setup selesai


In [12]:
# CELL 06.3 — Load income_features.csv
df_features = pd.read_csv('data/processed/income_features.csv')

with open('outputs/model_contract/feature_columns.json', encoding='utf-8') as f:
    feature_meta = json.load(f)

FEATURE_COLS = feature_meta['feature_columns']
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_features.columns]

print(f'Dimuat: income_features.csv ({df_features.shape})')
print(f'Feature cols: {len(FEATURE_COLS)}')
print('Target: next_week_income, next_week_direction')

required_cols = ['synthetic_user_id', 'next_week_income', 'next_week_direction']
missing_required = [c for c in required_cols if c not in df_features.columns]
assert not missing_required, f'Missing required columns: {missing_required}'


Dimuat: income_features.csv ((144000, 69))
Feature cols: 58
Target: next_week_income, next_week_direction


In [13]:
# CELL 06.4 — Kronologis split by synthetic_user_id (bukan random rows)
unique_ids = df_features['synthetic_user_id'].unique()
np.random.shuffle(unique_ids)

n = len(unique_ids)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

train_ids = unique_ids[:n_train]
val_ids   = unique_ids[n_train:n_train + n_val]
test_ids  = unique_ids[n_train + n_val:]

df_train = df_features[df_features['synthetic_user_id'].isin(train_ids)].copy()
df_val   = df_features[df_features['synthetic_user_id'].isin(val_ids)].copy()
df_test  = df_features[df_features['synthetic_user_id'].isin(test_ids)].copy()

print('Split by synthetic_user_id:')
print(f'  Train : {len(train_ids):,} users → {len(df_train):,} rows (70%)')
print(f'  Val   : {len(val_ids):,} users → {len(df_val):,} rows (15%)')
print(f'  Test  : {len(test_ids):,} users → {len(df_test):,} rows (15%)')

print('Leakage check user overlap: ', end='')
a = set(train_ids)
b = set(val_ids)
c = set(test_ids)
assert len(a & b) == 0 and len(a & c) == 0 and len(b & c) == 0, "Overlap detected!"
print('PASSED')


Split by synthetic_user_id:
  Train : 2,100 users → 100,800 rows (70%)
  Val   : 450 users → 21,600 rows (15%)
  Test  : 450 users → 21,600 rows (15%)
Leakage check user overlap: PASSED


In [14]:
# CELL 06.5 — Scaler (fit on train only)
X_train = df_train[FEATURE_COLS].fillna(0)
y_train = df_train['next_week_income'].values

target_scaler = MinMaxScaler()
target_scaler.fit(np.log1p(y_train).reshape(-1, 1))

feat_scaler = RobustScaler()
feat_scaler.fit(X_train)

ensure_dir('outputs/preprocessors/')

with open('outputs/preprocessors/weekly_target_scaler.pkl', 'wb') as f:
    pickle.dump(target_scaler, f)

with open('outputs/preprocessors/weekly_feature_scaler.pkl', 'wb') as f:
    pickle.dump(feat_scaler, f)

scalers_bundle = {
    'target_scaler': target_scaler,
    'feature_scaler': feat_scaler,
}

with open('outputs/model_contract/income_scalers.pkl', 'wb') as f:
    pickle.dump(scalers_bundle, f)

train_norm = target_scaler.transform(np.log1p(y_train).reshape(-1, 1))

print('Target scaler: log1p → MinMaxScaler (fit on train only)')
print(f'Train norm range: {train_norm.min():.4f} – {train_norm.max():.4f}')
print('Saved: income_scalers.pkl, weekly_target_scaler.pkl, weekly_feature_scaler.pkl')


Target scaler: log1p → MinMaxScaler (fit on train only)
Train norm range: 0.0000 – 1.0000
Saved: income_scalers.pkl, weekly_target_scaler.pkl, weekly_feature_scaler.pkl


In [15]:
# CELL 06.6 — Simpan train/val/test
safe_to_csv(df_train, 'outputs/model_contract/income_train.csv')
safe_to_csv(df_val,   'outputs/model_contract/income_val.csv')
safe_to_csv(df_test,  'outputs/model_contract/income_test.csv')

# Simpan juga ke data/synthetic untuk backward compatibility
safe_to_csv(df_train, 'data/synthetic/synthetic_52w_train.csv')
safe_to_csv(df_test,  'data/synthetic/synthetic_52w_test.csv')

print('Disimpan:')
print(f'  outputs/model_contract/income_train.csv  ({len(df_train):,} rows)')
print(f'  outputs/model_contract/income_val.csv    ({len(df_val):,} rows)')
print(f'  outputs/model_contract/income_test.csv   ({len(df_test):,} rows)')
print('  data/synthetic/synthetic_52w_train.csv')
print('  data/synthetic/synthetic_52w_test.csv')


Disimpan:
  outputs/model_contract/income_train.csv  (100,800 rows)
  outputs/model_contract/income_val.csv    (21,600 rows)
  outputs/model_contract/income_test.csv   (21,600 rows)
  data/synthetic/synthetic_52w_train.csv
  data/synthetic/synthetic_52w_test.csv


In [16]:
# CELL 06.7 — Simpan model_contract.json
model_contract = {
    'version': 'v13-FINAL',
    'created': '2026-05',
    'team': 'CC26-PSU217',
    'pipeline': 'Notebook 01→02→03→04→05→06→07→08',
    'feature_columns': FEATURE_COLS,
    'n_features': len(FEATURE_COLS),
    'target_regression': 'next_week_income',
    'target_classification': 'next_week_direction',
    'direction_threshold': '10% (>= Up, <= Down)',
    'income_sequence_note': 'income_w4=terlama(lag_4), income_w1=terbaru(lag_1). Urutan: w4→w3→w2→w1',
    'split_rule': 'Kronologis by synthetic_user_id: 70% train / 15% val / 15% test',
    'normalization': 'log1p → MinMaxScaler (fit on train only)',
    'scaler_file': 'outputs/model_contract/income_scalers.pkl',
    'monthly_estimation': 'Agregasi 4 prediksi mingguan',
    'forbidden_leakage_cols': [
        'next_week_income',
        'next_week_direction',
        'monthly_income',
        'avg_weekly_income',
        'income_std_4w',
        'income_cv_4w',
        'income_range_4w',
        'income_w1',
        'income_w2',
        'income_w3',
        'income_w4',
        'synthetic_weekly_income',
    ],
    'data_summary': {
        'n_train_users': int(len(train_ids)),
        'n_val_users': int(len(val_ids)),
        'n_test_users': int(len(test_ids)),
        'n_train_rows': int(len(df_train)),
        'n_val_rows': int(len(df_val)),
        'n_test_rows': int(len(df_test)),
    }
}

with open('outputs/model_contract/model_contract.json', 'w', encoding='utf-8') as f:
    json.dump(model_contract, f, indent=2, ensure_ascii=False)

print('Disimpan: outputs/model_contract/model_contract.json')
print('\n=== Model Contract Summary ===')
for k, v in model_contract.items():
    if not isinstance(v, list):
        print(f'  {k}: {v}')


Disimpan: outputs/model_contract/model_contract.json

=== Model Contract Summary ===
  version: v13-FINAL
  created: 2026-05
  team: CC26-PSU217
  pipeline: Notebook 01→02→03→04→05→06→07→08
  n_features: 58
  target_regression: next_week_income
  target_classification: next_week_direction
  direction_threshold: 10% (>= Up, <= Down)
  income_sequence_note: income_w4=terlama(lag_4), income_w1=terbaru(lag_1). Urutan: w4→w3→w2→w1
  split_rule: Kronologis by synthetic_user_id: 70% train / 15% val / 15% test
  normalization: log1p → MinMaxScaler (fit on train only)
  scaler_file: outputs/model_contract/income_scalers.pkl
  monthly_estimation: Agregasi 4 prediksi mingguan
  data_summary: {'n_train_users': 2100, 'n_val_users': 450, 'n_test_users': 450, 'n_train_rows': 100800, 'n_val_rows': 21600, 'n_test_rows': 21600}


In [17]:
# CELL 06.8 — Final validation untuk AI Engineer
print('=== Final Model Dataset Validation ===')
print(f'Train rows: {len(df_train):,}')
print(f'Val rows  : {len(df_val):,}')
print(f'Test rows : {len(df_test):,}')
print(f'Feature columns: {len(FEATURE_COLS)}')

print('\nTarget income summary by split:')
for name, dfx in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f'\n{name.upper()}')
    print(dfx['next_week_income'].describe().round(0).to_string())

print('\nDirection distribution by split:')
for name, dfx in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f'\n{name.upper()}')
    print(dfx['next_week_direction'].value_counts(normalize=True).round(3).to_string())


=== Final Model Dataset Validation ===
Train rows: 100,800
Val rows  : 21,600
Test rows : 21,600
Feature columns: 58

Target income summary by split:

TRAIN
count     100800.0
mean      398780.0
std       306215.0
min           28.0
25%       182436.0
50%       318641.0
75%       517383.0
max      1940300.0

VAL
count      21600.0
mean      396105.0
std       289558.0
min           15.0
25%       182225.0
50%       325656.0
75%       527451.0
max      1940300.0

TEST
count      21600.0
mean      393519.0
std       301705.0
min          153.0
25%       179580.0
50%       320931.0
75%       510247.0
max      1940300.0

Direction distribution by split:

TRAIN
next_week_direction
Stable    0.641
Up        0.239
Down      0.120

VAL
next_week_direction
Stable    0.650
Up        0.233
Down      0.116

TEST
next_week_direction
Stable    0.647
Up        0.235
Down      0.118


In [18]:
# GIT PUSH — Commit dan push output notebook ini ke GitHub
import os, subprocess

LOCAL_DIR   = "/content/fingo-income-analysis"
BRANCH_NAME = "feat/income-predictor-final"
NOTEBOOK_NAME = "06_Model_Dataset_Split.ipynb"

os.chdir(LOCAL_DIR)

def run_cmd(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"$ {cmd}")
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command gagal: {cmd}")
    return r

run_cmd('git config user.email "adelineclarisya@gmail.com"')
run_cmd('git config user.name "ClarisyaA"')

print("\n[1] Cek status")
run_cmd("git status --short", check=False)

print("\n[2] Add semua perubahan output")
run_cmd("git add data/ outputs/ notebooks/ *.ipynb", check=False)

print("\n[3] Commit")
commit_result = run_cmd(
    f'git commit -m "feat(DS2): output dari {NOTEBOOK_NAME}"',
    check=False
)
if commit_result.returncode != 0:
    print("[INFO] Tidak ada perubahan baru, skip commit.")

print("\n[4] Fetch remote terbaru")
run_cmd("git fetch origin")

print("\n[5] Rebase lalu push")
run_cmd(f"git pull --rebase origin {BRANCH_NAME}")
run_cmd(f"git push -u origin {BRANCH_NAME}")

print("\nPush berhasil!")


$ git config user.email "adelineclarisya@gmail.com"
$ git config user.name "ClarisyaA"

[1] Cek status
$ git status --short

[2] Add semua perubahan output
$ git add data/ outputs/ notebooks/ *.ipynb

[3] Commit
$ git commit -m "feat(DS2): output dari 06_Model_Dataset_Split.ipynb"
On branch feat/income-predictor-final
Your branch is up to date with 'origin/feat/income-predictor-final'.

nothing to commit, working tree clean
[INFO] Tidak ada perubahan baru, skip commit.

[4] Fetch remote terbaru
$ git fetch origin

[5] Rebase lalu push
$ git pull --rebase origin feat/income-predictor-final
Already up to date.
From https://github.com/ClarisyaA/fingo-income-analysis
 * branch            feat/income-predictor-final -> FETCH_HEAD
$ git push -u origin feat/income-predictor-final
Branch 'feat/income-predictor-final' set up to track remote branch 'feat/income-predictor-final' from 'origin'.
Everything up-to-date

Push berhasil!
